# Example: a mixture of different families

Response times of a service: most requests are served from a cache and follow an exponential law, a database query
gives a normal peak around 60 ms, and a background of timeouts and retries is spread uniformly up to 200 ms. The
families have different supports, and a partitioner does not know them: the question is which partitioner gives a
start from which EM finds the truth.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
from pysatl_core.families import ParametricFamilyRegister
from pysatl_core.types import FamilyName

from pysatl_mpest.core import MixtureModel
from pysatl_mpest.estimators import EM
from pysatl_mpest.estimators.iterative import StepBreakpointer
from pysatl_mpest.optimizers import ScipyNelderMead

normal = ParametricFamilyRegister.get(FamilyName.NORMAL)
expon = ParametricFamilyRegister.get(FamilyName.EXPONENTIAL)
uniform = ParametricFamilyRegister.get(FamilyName.CONTINUOUS_UNIFORM)


def show(mixture: MixtureModel) -> None:
    for weight, component in zip(mixture.weights, mixture):
        parameters = {name: round(value, 3) for name, value in component.parameters.items()}
        print(f"weight={weight:.3f}  {component.family_name} {parameters}")


def em(steps: int) -> EM:
    return EM(breakpointers=[StepBreakpointer(steps)], pruners=[], optimizer=ScipyNelderMead())


def plot_mixtures(ax, X, mixtures, bins=60):
    grid = np.linspace(X.min(), X.max(), 1000)
    ax.hist(X, bins=bins, density=True, color="lightgray")
    for label, mixture in mixtures.items():
        ax.plot(grid, np.exp(mixture.lpdf(grid)), label=label)
    ax.legend()

from sklearn.cluster import AgglomerativeClustering, KMeans

from pysatl_mpest.initializers import (
    ClusterInitializer,
    InitializationError,
    KMeansPartitioner,
    MultiStartInitializer,
    QuantilePartitioner,
    RandomCentersPartitioner,
    RefiningInitializer,
    SklearnPartitioner,
)

## Data

In [ ]:
truth = MixtureModel(
    [expon(lambda_=1 / 15), normal(mu=60, sigma=6), uniform(lower_bound=0, upper_bound=200)], weights=[0.5, 0.35, 0.15]
)
X = truth.generate(size=600, random_state=42)
families = [expon, normal, uniform]

fig, ax = plt.subplots(figsize=(8, 3.5))
plot_mixtures(ax, X, {"true mixture": truth}, bins=80)
ax.set_title("Response times, ms")
plt.show()

## Starts of five partitioners

Every partitioner gets five seeds; every start is refined by 20 EM iterations.

In [ ]:
partitioners = {
    "k-means": KMeansPartitioner,
    "quantile": QuantilePartitioner,
    "random centers": RandomCentersPartitioner,
    "sklearn KMeans": lambda: SklearnPartitioner(KMeans(n_init=1)),
    "agglomerative": lambda: SklearnPartitioner(AgglomerativeClustering()),
}
results = {}
for name, make in partitioners.items():
    scores = []
    for seed in range(5):
        try:
            start = ClusterInitializer(make()).initialize(X, families, random_state=seed)
        except InitializationError:
            continue
        fitted = em(20).fit(X, start)
        scores.append(fitted.loglikelihood(X))
        if seed == 0:
            results[name] = (start, fitted)
    print(f"{name:15} successful starts {len(scores)}/5, log-likelihood after EM from {min(scores):.1f} to {max(scores):.1f}")
print("true mixture:", round(truth.loglikelihood(X), 1))

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(20, 3.5), sharey=True)
for ax, (name, (start, fitted)) in zip(axes, results.items()):
    plot_mixtures(ax, X, {"start": start, "after EM": fitted}, bins=80)
    ax.set_title(f"{name}: {fitted.loglikelihood(X):.0f}")
plt.show()

## What happened

- The quantile split cuts the dense exponential part into two clusters and puts the normal peak together with the
  background. The uniform component starts on a wrong range, and EM cannot fix it: the support of a uniform component
  can only shrink during EM.
- k-means, its scikit-learn version and agglomerative clustering separate the peak, but give the uniform component
  only the right tail of the sample. EM comes close to the truth, but the uniform component stays on the tail.
- Random centers depend on the draw: some draws end like the quantile split, and some give the uniform component a
  wider range than the tail — then EM reaches a fit as good as the true mixture.

With different supports no single partitioner is safe, but the draws of random centers can be compared: a multistart
over refined runs keeps the draw with the best fit.

In [ ]:
refined = RefiningInitializer(ClusterInitializer(RandomCentersPartitioner()), em(20))
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    best = MultiStartInitializer(refined, n_init=5).initialize(X, families, random_state=0)
show(best)
print("multistart over refined random centers:", round(best.loglikelihood(X), 1), " k-means:", round(results["k-means"][1].loglikelihood(X), 1))

assert results["k-means"][1].loglikelihood(X) > results["quantile"][1].loglikelihood(X) + 100
assert best.loglikelihood(X) > results["k-means"][1].loglikelihood(X)